In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 1) Load data
def load_data(path_EMT, path_DP, path_SP):
    df_EMT = pd.read_csv(path_EMT)
    df_DP  = pd.read_csv(path_DP)
    df_SP  = pd.read_csv(path_SP)

    # Strip spaces from column names
    df_EMT.columns = df_EMT.columns.str.strip()
    df_DP.columns  = df_DP.columns.str.strip()
    df_SP.columns  = df_SP.columns.str.strip()

    print("EMT columns:", df_EMT.columns)
    print("DP columns:", df_DP.columns)
    print("SP columns:", df_SP.columns)

    return df_EMT, df_DP, df_SP

# 2) Compute base parameters
def compute_base_values():
    baseVoltageLineToLine   = 110e3
    baseVoltageLineToGround = baseVoltageLineToLine / np.sqrt(3)
    basePowerThreePhase     = 100e6
    baseCurrent             = basePowerThreePhase / baseVoltageLineToLine / np.sqrt(3)
    return baseCurrent, baseVoltageLineToLine

def compute_base_values_inverter():
    baseVoltageLineToLine   = 1500
    baseVoltageLineToGround = baseVoltageLineToLine / np.sqrt(3)
    basePowerThreePhase     = 100e6
    baseCurrent             = basePowerThreePhase / baseVoltageLineToLine / np.sqrt(3)
    baseCurrent             = baseCurrent * np.sqrt(3)  # account for Park transformation
    return baseCurrent, baseVoltageLineToLine

def compute_base_power():
    return 100e6

# Helper: pick the first existing column name
def pick_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

# ----- PLOT FUNCTIONS -----
def plot_load(df_EMT, df_DP, df_SP, baseCurrent, baseVoltage):
    plt.figure(1, figsize=(12, 6))
    # Subplot 1: Load currents
    plt.subplot(2, 1, 1)
    plt.grid(True)
    EMT_load_I0 = (df_EMT['iLoad1_0'] + df_EMT['iLoad2_0']) / baseCurrent / np.sqrt(2)
    EMT_load_I1 = (df_EMT['iLoad1_1'] + df_EMT['iLoad2_1']) / baseCurrent / np.sqrt(2)
    EMT_load_I2 = (df_EMT['iLoad1_2'] + df_EMT['iLoad2_2']) / baseCurrent / np.sqrt(2)
    plt.plot(df_EMT['time'], EMT_load_I0, label='EMT Iload Ph1')
    plt.plot(df_EMT['time'], EMT_load_I1, label='EMT Iload Ph2')
    plt.plot(df_EMT['time'], EMT_load_I2, label='EMT Iload Ph3')

    Iload_DP = np.abs((df_DP['iLoad1.re'] + df_DP['iLoad2.re']) + 1j*(df_DP['iLoad1.im'] + df_DP['iLoad2.im'])) / baseCurrent / np.sqrt(3)
    Iload_SP = np.abs((df_SP['iLoad1.re'] + df_SP['iLoad2.re']) + 1j*(df_SP['iLoad1.im'] + df_SP['iLoad2.im'])) / baseCurrent / np.sqrt(3)
    plt.plot(df_DP['time'], Iload_DP, 'r--', label='DP Iload')
    plt.plot(df_SP['time'], Iload_SP, 'k--', label='RMS Iload')
    plt.xlabel('Time (s)')
    plt.ylabel('Load Current (p.u.)')
    plt.xlim(3.78, 3.82)
    plt.legend(loc='lower left')

    # Subplot 2: Load voltages
    plt.subplot(2, 1, 2)
    plt.grid(True)
    v0 = df_EMT['vLoad_0'] / baseVoltage * np.sqrt(3) / np.sqrt(2)
    v1 = df_EMT['vLoad_1'] / baseVoltage * np.sqrt(3) / np.sqrt(2)
    v2 = df_EMT['vLoad_2'] / baseVoltage * np.sqrt(3) / np.sqrt(2)
    plt.plot(df_EMT['time'], v0, label='EMT Vload Ph1')
    plt.plot(df_EMT['time'], v1, label='EMT Vload Ph2')
    plt.plot(df_EMT['time'], v2, label='EMT Vload Ph3')

    Vload_DP = np.abs(df_DP['vLoad.re'] + 1j*df_DP['vLoad.im']) / baseVoltage
    Vload_SP = np.abs(df_SP['vLoad.re'] + 1j*df_SP['vLoad.im']) / baseVoltage
    plt.plot(df_DP['time'], Vload_DP, 'r--', label='DP Vload')
    plt.plot(df_SP['time'], Vload_SP, 'k--', label='RMS Vload')
    plt.xlabel('Time (s)')
    plt.ylabel('Load Voltage (p.u.)')
    plt.xlim(3.78, 3.82)
    plt.legend(loc='lower left')


def plot_infeed(df_EMT, df_DP, df_SP, baseCurrent, baseVoltage):
    plt.figure(2, figsize=(12, 6))
    plt.subplot(2, 1, 1)
    plt.grid(True)
    plt.plot(df_EMT['time'], -df_EMT['iInfeed_0'] / baseCurrent / np.sqrt(2), label='EMT Iinfeed Ph1')
    plt.plot(df_EMT['time'], -df_EMT['iInfeed_1'] / baseCurrent / np.sqrt(2), label='EMT Iinfeed Ph2')
    plt.plot(df_EMT['time'], -df_EMT['iInfeed_2'] / baseCurrent / np.sqrt(2), label='EMT Iinfeed Ph3')

    Iinf_DP = np.abs(df_DP['iInfeed.re'] + 1j * df_DP['iInfeed.im']) / baseCurrent / np.sqrt(3)
    Iinf_SP = np.abs(df_SP['iInfeed.re'] + 1j * df_SP['iInfeed.im']) / baseCurrent / np.sqrt(3)
    plt.plot(df_DP['time'], Iinf_DP, 'r--', label='DP Iinfeed')
    plt.plot(df_SP['time'], Iinf_SP, 'k--', label='RMS Iinfeed')
    plt.xlabel('Time (s)')
    plt.ylabel('Infeed Current (p.u.)')
    plt.xlim(3.78, 3.82)
    plt.legend(loc='lower left')

    plt.subplot(2, 1, 2)
    plt.grid(True)
    plt.plot(df_EMT['time'], df_EMT['vInfeed_0']/baseVoltage * np.sqrt(3) / np.sqrt(2), label='EMT Vinfeed Ph1')
    plt.plot(df_EMT['time'], df_EMT['vInfeed_1']/baseVoltage * np.sqrt(3) / np.sqrt(2), label='EMT Vinfeed Ph2')
    plt.plot(df_EMT['time'], df_EMT['vInfeed_2']/baseVoltage * np.sqrt(3) / np.sqrt(2), label='EMT Vinfeed Ph3')

    Vinf_DP = np.abs(df_DP['vInfeed.re'] + 1j * df_DP['vInfeed.im']) / baseVoltage
    Vinf_SP = np.abs(df_SP['vInfeed.re'] + 1j * df_SP['vInfeed.im']) / baseVoltage
    plt.plot(df_DP['time'], Vinf_DP, 'r--', label='DP Vinfeed')
    plt.plot(df_SP['time'], Vinf_SP, 'k--', label='RMS Vinfeed')
    plt.xlabel('Time (s)')
    plt.ylabel('Infeed Voltage (p.u.)')
    plt.xlim(3.78, 3.82)
    plt.legend(loc='lower left')


def plot_converter1(df_EMT, df_DP, df_SP, baseCurrent, baseVoltage):
    plt.figure(3, figsize=(12, 6))
    plt.subplot(2, 1, 1)
    plt.grid(True)
    EMT_I1_0 = df_EMT['iLine1_0'] / baseCurrent / np.sqrt(2)
    EMT_I1_1 = df_EMT['iLine1_1'] / baseCurrent / np.sqrt(2)
    EMT_I1_2 = df_EMT['iLine1_2'] / baseCurrent / np.sqrt(2)
    plt.plot(df_EMT['time'], EMT_I1_0, label='EMT ILine1 Ph1')
    plt.plot(df_EMT['time'], EMT_I1_1, label='EMT ILine1 Ph2')
    plt.plot(df_EMT['time'], EMT_I1_2, label='EMT ILine1 Ph3')

    I1_DP = np.abs(df_DP['iLine1.re'] + 1j * df_DP['iLine1.im']) / baseCurrent / np.sqrt(3)
    I1_SP = np.abs(df_SP['iLine1.re'] + 1j * df_SP['iLine1.im']) / baseCurrent / np.sqrt(3)
    plt.plot(df_DP['time'], I1_DP, 'r--', label='DP ILine1')
    plt.plot(df_SP['time'], I1_SP, 'k--', label='RMS ILine1')
    plt.xlabel('Time (s)')
    plt.ylabel('Line1 Current (p.u.)')
    plt.xlim(3.78, 3.82)
    plt.legend(loc='lower left')

    plt.subplot(2, 1, 2)
    plt.grid(True)
    V1_0 = df_EMT['vConverter1_0'] / baseVoltage * np.sqrt(3) / np.sqrt(2)
    V1_1 = df_EMT['vConverter1_1'] / baseVoltage * np.sqrt(3) / np.sqrt(2)
    V1_2 = df_EMT['vConverter1_2'] / baseVoltage * np.sqrt(3) / np.sqrt(2)
    plt.plot(df_EMT['time'], V1_0, label='EMT Vconv1 Ph1')
    plt.plot(df_EMT['time'], V1_1, label='EMT Vconv1 Ph2')
    plt.plot(df_EMT['time'], V1_2, label='EMT Vconv1 Ph3')

    V1_DP = np.abs(df_DP['vConverter1.re'] + 1j * df_DP['vConverter1.im']) / baseVoltage
    V1_SP = np.abs(df_SP['vConverter1.re'] + 1j * df_SP['vConverter1.im']) / baseVoltage
    plt.plot(df_DP['time'], V1_DP, 'r--', label='DP Vconv1')
    plt.plot(df_SP['time'], V1_SP, 'k--', label='RMS Vconv1')
    plt.xlabel('Time (s)')
    plt.ylabel('Converter1 Voltage (p.u.)')
    plt.xlim(3.78, 3.82)
    plt.legend(loc='lower left')

def plot_converter2(df_EMT, df_DP, df_SP, baseCurrent, baseVoltage):
    plt.figure(4, figsize=(12, 6))
    plt.subplot(2, 1, 1)
    plt.grid(True)
    EMT_I2_0 = df_EMT['iLine2_0'] / baseCurrent / np.sqrt(2)
    EMT_I2_1 = df_EMT['iLine2_1'] / baseCurrent / np.sqrt(2)
    EMT_I2_2 = df_EMT['iLine2_2'] / baseCurrent / np.sqrt(2)
    plt.plot(df_EMT['time'], EMT_I2_0, label='EMT Iconv2_0')
    plt.plot(df_EMT['time'], EMT_I2_1, label='EMT Iconv2_1')
    plt.plot(df_EMT['time'], EMT_I2_2, label='EMT Iconv2_2')

    I2_DP = np.abs(df_DP['iLine2.re'] + 1j * df_DP['iLine2.im']) / baseCurrent / np.sqrt(3)
    I2_SP = np.abs(df_SP['iLine2.re'] + 1j * df_SP['iLine2.im']) / baseCurrent / np.sqrt(3)
    plt.plot(df_DP['time'], I2_DP, 'r--', label='DP Iconv2')
    plt.plot(df_SP['time'], I2_SP, 'k--', label='RMS Iconv2')
    plt.xlabel('Time (s)')
    plt.ylabel('Line2 Current (p.u.)')
    plt.xlim(3.78, 3.82)
    plt.legend(loc='lower left')

    plt.subplot(2, 1, 2)
    plt.grid(True)
    V2_0 = df_EMT['vConverter2_0'] / baseVoltage * np.sqrt(3) / np.sqrt(2)
    V2_1 = df_EMT['vConverter2_1'] / baseVoltage * np.sqrt(3) / np.sqrt(2)
    V2_2 = df_EMT['vConverter2_2'] / baseVoltage * np.sqrt(3) / np.sqrt(2)
    plt.plot(df_EMT['time'], V2_0, label='EMT Vconv2 Ph1')
    plt.plot(df_EMT['time'], V2_1, label='EMT Vconv2 Ph2')
    plt.plot(df_EMT['time'], V2_2, label='EMT Vconv2 Ph3')

    V2_DP = np.abs(df_DP['vConverter2.re'] + 1j * df_DP['vConverter2.im']) / baseVoltage
    V2_SP = np.abs(df_SP['vConverter2.re'] + 1j * df_SP['vConverter2.im']) / baseVoltage
    plt.plot(df_DP['time'], V2_DP, 'r--', label='DP Vconv2')
    plt.plot(df_SP['time'], V2_SP, 'k--', label='RMS Vconv2')
    plt.xlabel('Time (s)')
    plt.ylabel('Converter2 Voltage (p.u.)')
    plt.xlim(3.78, 3.82)
    plt.legend(loc='lower left')

def plot_converter_internal_vars(df_EMT, df_DP, df_SP, baseCurrent, baseVoltage, conv_idx):
    fig_num = 4 + conv_idx  # 5 for conv1, 6 for conv2
    plt.figure(fig_num, figsize=(12, 6))
    colors = {'EMT': 'blue', 'DP': 'red', 'SP': 'black'}

    id_col = f'idConverter{conv_idx}'
    iq_col = f'iqConverter{conv_idx}'
    vd_col = f'vdConverter{conv_idx}'
    vq_col = f'vqConverter{conv_idx}'

    plt.subplot(4, 1, 1)
    plt.grid(True)
    plt.plot(df_EMT['time'], df_EMT[id_col] / baseCurrent, label='EMT Id', linestyle='-', color=colors['EMT'])
    plt.plot(df_DP['time'],  df_DP[id_col]  / baseCurrent, label='DP Id',  linestyle='--', color=colors['DP'])
    plt.plot(df_SP['time'],  df_SP[id_col]  / baseCurrent, label='RMS Id', linestyle=':', color=colors['SP'])
    plt.ylabel('d-axis Current (p.u.)')
    plt.xlim(3.78, 3.82)
    plt.legend(loc='lower left')

    plt.subplot(4, 1, 2)
    plt.grid(True)
    plt.plot(df_EMT['time'], df_EMT[iq_col] / baseCurrent, label='EMT Iq', linestyle='-', color=colors['EMT'])
    plt.plot(df_DP['time'],  df_DP[iq_col]  / baseCurrent, label='DP Iq',  linestyle='--', color=colors['DP'])
    plt.plot(df_SP['time'],  df_SP[iq_col]  / baseCurrent, label='RMS Iq', linestyle=':', color=colors['SP'])
    plt.ylabel('q-axis Current (p.u.)')
    plt.xlim(3.78, 3.82)
    plt.legend(loc='lower left')

    plt.subplot(4, 1, 3)
    plt.grid(True)
    plt.plot(df_EMT['time'], df_EMT[vd_col] / baseVoltage, label='EMT Vd', linestyle='-', color=colors['EMT'])
    plt.plot(df_DP['time'],  df_DP[vd_col]  / baseVoltage, label='DP Vd',  linestyle='--', color=colors['DP'])
    plt.plot(df_SP['time'],  df_SP[vd_col]  / baseVoltage, label='RMS Vd', linestyle=':', color=colors['SP'])
    plt.ylabel('d-axis Voltage (p.u.)')
    plt.xlim(3.78, 3.82)
    plt.legend(loc='lower left')

    plt.subplot(4, 1, 4)
    plt.grid(True)
    plt.plot(df_EMT['time'], df_EMT[vq_col] / baseVoltage, label='EMT Vq', linestyle='-', color=colors['EMT'])
    plt.plot(df_DP['time'],  df_DP[vq_col]  / baseVoltage, label='DP Vq',  linestyle='--', color=colors['DP'])
    plt.plot(df_SP['time'],  df_SP[vq_col]  / baseVoltage, label='RMS Vq', linestyle=':', color=colors['SP'])
    plt.xlabel('Time (s)')
    plt.ylabel('q-axis Voltage (p.u.)')
    plt.xlim(3.78, 3.82)
    plt.legend(loc='lower left')

def plot_power_refs(df_EMT, df_DP, df_SP, basePowerThreePhase, conv_idx=1):
    """
    Plots the additionally logged Pref/Qref (e.g., PrefConverter1, QrefConverter1) in p.u.
    """
    plt.figure(7 + (conv_idx - 1), figsize=(12, 5))

    pref_name = f'PrefConverter{conv_idx}'
    qref_name = f'QrefConverter{conv_idx}'

    # be tolerant if someone logged ".re" or similar
    EMT_pref = pick_col(df_EMT, [pref_name, pref_name + ".re"])
    DP_pref  = pick_col(df_DP,  [pref_name, pref_name + ".re"])
    SP_pref  = pick_col(df_SP,  [pref_name, pref_name + ".re"])

    EMT_qref = pick_col(df_EMT, [qref_name, qref_name + ".re"])
    DP_qref  = pick_col(df_DP,  [qref_name, qref_name + ".re"])
    SP_qref  = pick_col(df_SP,  [qref_name, qref_name + ".re"])

    if EMT_pref is None or DP_pref is None or SP_pref is None:
        print(f"[WARN] Missing Pref columns for converter {conv_idx}:",
              EMT_pref, DP_pref, SP_pref)
    if EMT_qref is None or DP_qref is None or SP_qref is None:
        print(f"[WARN] Missing Qref columns for converter {conv_idx}:",
              EMT_qref, DP_qref, SP_qref)

    plt.subplot(2, 1, 1)
    plt.grid(True)
    if EMT_pref: plt.plot(df_EMT['time'], df_EMT[EMT_pref] / basePowerThreePhase, label='EMT P_ref')
    if DP_pref:  plt.plot(df_DP['time'],  df_DP[DP_pref]  / basePowerThreePhase, 'r--', label='DP P_ref')
    if SP_pref:  plt.plot(df_SP['time'],  df_SP[SP_pref]  / basePowerThreePhase, 'k:',  label='RMS P_ref')
    plt.ylabel('P_ref (p.u.)')
    plt.xlim(3.78, 3.82)
    plt.legend(loc='lower left')

    plt.subplot(2, 1, 2)
    plt.grid(True)
    if EMT_qref: plt.plot(df_EMT['time'], df_EMT[EMT_qref] / basePowerThreePhase, label='EMT Q_ref')
    if DP_qref:  plt.plot(df_DP['time'],  df_DP[DP_qref]  / basePowerThreePhase, 'r--', label='DP Q_ref')
    if SP_qref:  plt.plot(df_SP['time'],  df_SP[SP_qref]  / basePowerThreePhase, 'k:',  label='RMS Q_ref')
    plt.xlabel('Time (s)')
    plt.ylabel('Q_ref (p.u.)')
    plt.xlim(3.78, 3.82)
    plt.legend(loc='lower left')

def plot_fInfeed(df_EMT, df_DP, df_SP):
    plt.figure(figsize=(12, 4))
    plt.grid(True)
    plt.plot(df_EMT['time'], df_EMT['fInfeed'], label='EMT fInfeed', color='blue', linestyle='-')
    plt.plot(df_DP['time'], df_DP['fInfeed'], label='DP fInfeed', color='red', linestyle='--')
    plt.plot(df_SP['time'], df_SP['fInfeed'], label='SP fInfeed', color='black', linestyle=':')
    plt.xlabel('Time (s)')
    plt.ylabel('Infeed Frequency (Hz)')
    plt.xlim(3.78, 3.82)
    plt.ylim(48, 51)
    plt.legend(loc='lower left')

def main():
    # Change paths here
    path_EMT = "/home/gti/development/dpsim_local/dpsim/build/dpsim/examples/cxx/logs/EMT_simulation/EMT_simulation.csv"
    path_DP  = "/home/gti/development/dpsim_local/dpsim/build/dpsim/examples/cxx/logs/DP_simulation/DP_simulation.csv"
    path_SP  = "/home/gti/development/dpsim_local/dpsim/build/dpsim/examples/cxx/logs/SP_simulation/SP_simulation.csv"

    df_EMT, df_DP, df_SP = load_data(path_EMT, path_DP, path_SP)
    baseCurrent, baseVoltage = compute_base_values()
    baseCurrentInverter, baseVoltageInverter = compute_base_values_inverter()
    basePower = compute_base_power()

    plot_load(df_EMT, df_DP, df_SP, baseCurrent, baseVoltage)
    plot_infeed(df_EMT, df_DP, df_SP, baseCurrent, baseVoltage)
    plot_converter1(df_EMT, df_DP, df_SP, baseCurrent, baseVoltage)
    plot_converter2(df_EMT, df_DP, df_SP, baseCurrent, baseVoltage)

    plot_converter_internal_vars(df_EMT, df_DP, df_SP, baseCurrentInverter, baseVoltageInverter, conv_idx=1)
    plot_converter_internal_vars(df_EMT, df_DP, df_SP, baseCurrentInverter, baseVoltageInverter, conv_idx=2)

    # NEW: plot logged power references (Pref/Qref)
    plot_power_refs(df_EMT, df_DP, df_SP, basePower, conv_idx=1)
    plot_power_refs(df_EMT, df_DP, df_SP, basePower, conv_idx=2)

    plot_fInfeed(df_EMT, df_DP, df_SP)
    plt.show()

if __name__ == "__main__":
    main()
